In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import string
import matplotlib.dates as mdates

from scipy.optimize import curve_fit
from scipy.optimize import minimize
from scipy.optimize import basinhopping

from scipy import stats

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
plt.rcParams['font.family']='Arial'

bar_width=0.25
plt.rc('axes', labelsize=10)
plt.rc('xtick', labelsize=10) 
plt.rc('ytick', labelsize=10) 
plt.rc('legend', fontsize=10)
plt.rc('figure', titlesize=12) 
alphsize=12
alpha=0.4
padsize=15
set_dpi=600

## Data

In [2]:
data_ori=pd.read_excel('./data/heat_region.xlsx')

In [3]:
data_p2=data_ori[data_ori['year']>=2018]
data_p2.reset_index(drop=True, inplace=True)

In [4]:
data_loc2=pd.DataFrame([],columns=['date','case','tavg','tmax','rhum','wbgt'])

In [5]:
data_loc2['date']=data_p2['date'].unique()
data_loc2['case']=data_p2.groupby('date')['case'].sum(min_count=1).values
data_loc2['tavg']=data_p2.groupby('date')['tavg'].mean().values
data_loc2['tmax']=data_p2.groupby('date')['tmax'].max().values
data_loc2['rhum']=data_p2.groupby('date')['rhum'].mean().values
data_loc2['wbgt']=data_p2.groupby('date')['wbgt'].max().values
data_loc2['year']=data_loc2['date'].dt.year
data_loc2['month']=data_loc2['date'].dt.month

In [6]:
data_loc2['day']=data_loc2['date'].dt.day
data_loc2['late']=0
data_loc2.loc[data_loc2['day']>15,'late']=1
data_loc2['month2']=data_loc2['month']+data_loc2['late']*0.5

In [7]:
data_loc2['phase']=0
data_loc2.loc[data_loc2['month2']>=8.5,'phase']=1

## Hockey stick model

In [8]:
def hockey_stick(x, x0, b1, b2):
    return np.where(x < x0, b1, b2 * (x - x0) + b1)

def log_likelihood(params, x,y):
    x0, b1, b2 = params
    y_pred = hockey_stick(x, x0, b1, b2)
    return -np.sum(stats.poisson.logpmf(y, y_pred))

initial_params = [30, 0, 13]

In [9]:
data_loc2_drop=data_loc2.dropna()
data_loc2_drop.reset_index(drop=True, inplace=True)

In [ ]:
data_loc2_zero=data_loc2.copy()
data_loc2_zero['case']=data_loc2_zero['case'].fillna(0)

In [ ]:
tmp=data_loc2_drop

minimizer_kwargs={"method":"SLSQP",
        "bounds":((tmp['wbgt'].min(),tmp['wbgt'].max()),(0,None),(0,None)),
        "args":(tmp['wbgt'], tmp['case'])}
    
result = basinhopping(log_likelihood, initial_params, minimizer_kwargs=minimizer_kwargs, niter=1000,niter_success=100)

t0,b1,b2=result.x
x_fit=np.linspace(21,38,100)
y_fit=hockey_stick(x_fit,t0,b1,b2)

In [17]:
num_params = len(result.x)  # Number of parameters (k)
log_likelihood_value = -log_likelihood(result.x, tmp['wbgt'], tmp['case'])
aic = 2 * num_params - 2 * log_likelihood_value

print(aic)

8597.814993967719


### Phase

In [ ]:
tmp1=data_loc2_drop.loc[data_loc2_drop['month2']<8.5]
tmp2=data_loc2_drop.loc[data_loc2_drop['month2']>=8.5]

minimizer_kwargs={"method":"SLSQP",
        "bounds":((tmp1['wbgt'].min(),tmp1['wbgt'].max()),(0,None),(None,None)),
        "args":(tmp1['wbgt'], tmp1['case'])}
    
result_p = basinhopping(log_likelihood, initial_params, minimizer_kwargs=minimizer_kwargs, niter=1000,niter_success=100)

t0_p,b1_p,b2_p=result_p.x
xp_fit=np.linspace(21,38,100)
yp_fit=hockey_stick(xp_fit,t0_p,b1_p,b2_p)

In [ ]:
minimizer_kwargs={"method":"SLSQP",
        "bounds":((tmp2['wbgt'].min(),tmp2['wbgt'].max()),(0,None),(None,None)),
        "args":(tmp2['wbgt'], tmp2['case'])}

result_a = basinhopping(log_likelihood, initial_params, minimizer_kwargs=minimizer_kwargs, niter=1000,niter_success=100)
        
t0_a,b1_a,b2_a=result_a.x
xa_fit=np.linspace(21,38,100)
ya_fit=hockey_stick(xa_fit,t0_a,b1_a,b2_a)

### Urban vs metro

In [25]:
data_region=data_p2.copy()

In [27]:
metro_data=pd.DataFrame([],columns=['date','case','wbgt'])
urban_data=pd.DataFrame([],columns=['date','case','wbgt'])

In [28]:
tmp_data=data_region.groupby('metro').get_group(1)
metro_data['date']=tmp_data['date'].unique()
metro_data['case']=tmp_data.groupby('date')['case'].sum(min_count=1).values
metro_data['wbgt']=tmp_data.groupby('date')['wbgt'].max().values

In [29]:
tmp_data=data_region.groupby('metro').get_group(0)
urban_data['date']=tmp_data['date'].unique()
urban_data['case']=tmp_data.groupby('date')['case'].sum(min_count=1).values
urban_data['wbgt']=tmp_data.groupby('date')['wbgt'].max().values

In [30]:
metro_data['month2']=metro_data['date'].dt.month+metro_data['date'].dt.day/30
urban_data['month2']=urban_data['date'].dt.month+urban_data['date'].dt.day/30

metro_data['phase']=0
metro_data.loc[metro_data['month2']>=8+16/30,'phase']=1

urban_data['phase']=0
urban_data.loc[urban_data['month2']>=8+16/30,'phase']=1

metro_data['year']=metro_data['date'].dt.year
urban_data['year']=urban_data['date'].dt.year

In [31]:
resultm_list=[]
ym_fit_list=[]

metro_data_drop=metro_data.dropna()
urban_data_drop=urban_data.dropna()

for x in [metro_data_drop,urban_data_drop]:
    tmp=x
    minimizer_kwargs={"method":"SLSQP",
        # "method":"Nelder-Mead",
        "bounds":((tmp['wbgt'].min(),tmp['wbgt'].max()),(0,None),(0,None)),
        "args":(tmp['wbgt'],tmp['case'])}
    
    resultm = basinhopping(log_likelihood, initial_params, minimizer_kwargs=minimizer_kwargs, niter=1000,niter_success=100)
    th1,b1,b2=resultm.x
    
    x_fit=np.linspace(21,38,100)
    ym_fit=hockey_stick(x_fit,th1,b1,b2)
    ym_fit_list.append(ym_fit)
    resultm_list.append(resultm.x)
    
    print(resultm.x)

[31.11381077  0.75376001  5.38067379]
[29.91899765  2.20575197 12.01572478]


### Region

In [ ]:
region_cap_name=['Seoul', 'Gyeonggi', 'Incheon', 'Chungbuk', 'Chungnam', 'Daejeon', 'Jeonbuk', 'Jeonnam', 'Gwangju', 'Jeju', 'Gangwon', 'Gyeongbuk', 'Daegu', 'Gyeongnam', 'Ulsan', 'Busan']
Region_code=['G1','G2','G3','G4','G5','G6','G7','G8','G9','G10', 'G11','G12','G13','G14','G15','G16']

In [ ]:
result_region_list=[]
y_fit_region_list=[]

data_region_drop=data_region.dropna()
data_region_drop.reset_index(drop=True, inplace=True)

for x in region_cap_name:
    tmp=data_region_drop[data_region_drop['region']==x]
    minimizer_kwargs={"method":"SLSQP",
        "bounds":((tmp['wbgt'].min(),tmp['wbgt'].max()),(0,None),(0,None)),
        "args":(tmp['wbgt'],tmp['case'])}
    
    result_region = basinhopping(log_likelihood, initial_params, minimizer_kwargs=minimizer_kwargs, niter=1000,niter_success=100)
    th1,b1,b2=result_region.x
    
    x_fit=np.linspace(21,38,100)
    y_fit_region=hockey_stick(x_fit,th1,b1,b2)
    y_fit_region_list.append(y_fit_region)
    result_region_list.append(result_region.x)
    
    print(x,result_region.x)

Seoul [29.59619658  0.24736847  2.58328438]
Gyeonggi [29.6760676   0.73636366  4.91171498]
Incheon [27.65807276  0.15360684  1.55694297]
Chungbuk [29.15373909  0.13407822  1.0413591 ]
Chungnam [29.03235221  0.2035225   1.4510032 ]
Daejeon [29.71588593  0.04715127  0.30780752]
Jeonbuk [28.86331069  0.15106945  1.05447182]
Jeonnam [28.50969159  0.31262953  1.52060293]
Gwangju [30.02642721  0.07407505  0.42182049]
Jeju [27.28498278  0.08703665  0.44373639]
Gangwon [28.42734627  0.17275886  0.99266089]
Gyeongbuk [28.27781918  0.26096041  1.30224772]
Daegu [29.61968123  0.04821748  0.37230329]
Gyeongnam [29.13312777  0.25581403  1.59706236]
Ulsan [28.36466117  0.05846765  0.37205982]
Busan [28.6539892   0.10299624  1.2333656 ]


### Forecasting

In [ ]:
data126=pd.read_csv('./data/SSP126_data.csv')
data245=pd.read_csv('./data/SSP245_data.csv')
data370=pd.read_csv('./data/SSP370_data.csv')
data585=pd.read_csv('./data/SSP585_data.csv')

In [44]:
data126['date']=pd.to_datetime(data126['date'])
data245['date']=pd.to_datetime(data245['date'])
data370['date']=pd.to_datetime(data370['date'])
data585['date']=pd.to_datetime(data585['date'])

In [ ]:
lstm126=pd.read_csv('./data/LSTM_result_SSP126.csv')
lstm245=pd.read_csv('./data/LSTM_result_SSP245.csv')
lstm370=pd.read_csv('./data/LSTM_result_SSP370.csv')
lstm585=pd.read_csv('./data/LSTM_result_SSP585.csv')

In [47]:
df_126=pd.DataFrame([],columns=['date','case','wbgt'])
df_126['date']=data126.date.unique()
df_126['wbgt']=data126.groupby('date')['wbgt'].max().values
df_126=df_126[df_126['date'].dt.strftime('%m-%d')>'05-21']
df_126=df_126[df_126['date'].dt.year<=2070]
df_126.reset_index(drop=True, inplace=True)
df_126['case']=lstm126['Korea']
df_126['year']=df_126['date'].dt.year

In [48]:
df_245=pd.DataFrame([],columns=['date','case','wbgt'])
df_245['date']=data245.date.unique()
df_245['wbgt']=data245.groupby('date')['wbgt'].max().values
df_245=df_245[df_245['date'].dt.strftime('%m-%d')>'05-21']
df_245=df_245[df_245['date'].dt.year<=2070]
df_245.reset_index(drop=True, inplace=True)
df_245['case']=lstm245['Korea']
df_245['year']=df_245['date'].dt.year

In [49]:
df_370=pd.DataFrame([],columns=['date','case','wbgt'])
df_370['date']=data370.date.unique()
df_370['wbgt']=data370.groupby('date')['wbgt'].max().values
df_370=df_370[df_370['date'].dt.strftime('%m-%d')>'05-21']
df_370=df_370[df_370['date'].dt.year<=2070]
df_370.reset_index(drop=True, inplace=True)
df_370['case']=lstm370['Korea']
df_370['year']=df_370['date'].dt.year

In [50]:
df_585=pd.DataFrame([],columns=['date','case','wbgt'])
df_585['date']=data585.date.unique()
df_585['wbgt']=data585.groupby('date')['wbgt'].max().values
df_585=df_585[df_585['date'].dt.strftime('%m-%d')>'05-21']
df_585=df_585[df_585['date'].dt.year<=2070]
df_585.reset_index(drop=True, inplace=True)
df_585['case']=lstm585['Korea']
df_585['year']=df_585['date'].dt.year

In [51]:
df_126=pd.concat([data_loc2_drop[['date','case','wbgt','year']],df_126])
df_126.reset_index(drop=True, inplace=True)

In [52]:
df_585=pd.concat([data_loc2_drop[['date','case','wbgt','year']],df_585])
df_585.reset_index(drop=True, inplace=True)

In [55]:
df_585['period']=-1
df_585.loc[df_585['year']<2024,'period']=0
df_585.loc[df_585['year']>2030,'period']=1
df_585.loc[df_585['year']>2040,'period']=2
df_585.loc[df_585['year']>2050,'period']=3
df_585.loc[df_585['year']>2060,'period']=4

In [ ]:
ssp_585_result=[]
ssp_585_fit=[]

for i in range(0,5):
        tmp=df_585[df_585['period']==i]

        minimizer_kwargs={"method":"SLSQP",

        "bounds":((tmp['wbgt'].min(),tmp['wbgt'].max()),(0,None),(0,None)),
        "args":(tmp['wbgt'], round(tmp['case']))}

        result_585 = basinhopping(log_likelihood, initial_params, minimizer_kwargs=minimizer_kwargs, niter=1000,niter_success=100)

        t0_ssp585,b1_ssp585,b2_ssp585=result_585.x
        x_fit_585=np.linspace(21,40,100)
        y_fit_585=hockey_stick(x_fit_585,t0_ssp585,b1_ssp585,b2_ssp585)
        ssp_585_result.append(result_585.x)
        ssp_585_fit.append(y_fit_585)

ssp_585_result

[array([30.65646323,  2.79249431, 16.14122548]),
 array([31.05315098,  6.6299566 , 28.5002073 ]),
 array([31.57497743,  7.4642851 , 34.652757  ]),
 array([31.96208011,  8.85521881, 39.40589924]),
 array([32.37727833,  9.26441253, 39.18586395])]